# Lab: recording Ground Station handovers with events (V5)
This is the V5 version of [handover_lab.ipynb](../v4/handover_lab.ipynb). Same scenario: **one constellation** (a single Walker Delta shell), **one Ground Station** in Toulouse, connected with the `best-angle-until-disconnection` strategy, and we record every handover.

The V4 lab **polls** the simulation: at each tick, it asks which satellite the Ground Station is connected to, and compares with the previous tick. In V5, the simulation **publishes** every change of the network on an event bus, so we just listen:
- `GSL_CREATED`: a ground station link is up. It also carries the **predicted disconnection time** of the link (`estimated_disconnection_time`), computed from the orbits when the link is created
- `GSL_UPDATED`: the link is still up; its elevation and distance changed
- `GSL_REMOVED`: the link is down

A handover is a `GSL_REMOVED` followed by a `GSL_CREATED` for the same Ground Station, in the same tick. On top of the V4 lab, we check the prediction: did each link drop when the simulation said it would?

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from sat_com_application.event_bus.bus import EventBus
from sat_com_application.event_bus.events import EventType, GSLCreatedEvent, GSLUpdatedEvent, GSLRemovedEvent
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty

## 1. One constellation, one Ground Station
The same constellation and Ground Station as the V4 lab. The Ground Station is written inline (`ground_objects`) instead of a data file: same Toulouse coordinates as [`single_ground_station.txt`](../v4/configurations/single_ground_station.txt).

We simulate 30 minutes, ticking every 15 seconds.

In [ ]:
TICK_SECONDS = 15

lab_config = {
    "simulation_name": "Handover Events Lab",
    "start_date": "2026-01-01 00:00:00.000000",
    "end_date": "2026-01-01 00:30:00.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "ground_objects_properties": [
        {
            "identifier": "Single Ground Station",
            "type": "ground_station",
            "ground_objects": [
                {"id": 0, "label": "Toulouse", "latitude": 43.6045, "longitude": 1.444, "altitude": 0.0}
            ],
            "connectivity_properties": {
                "elevation_above_horizon": 20,
                "ground_to_space_connections_strategy": "best-angle-until-disconnection",
            },
        },
    ],
    "walker_shells": [
        {
            "type": "delta",
            "constellation_property": {
                "identifier": "Handover Lab Walker",
                "amount_of_orbit_plane": 8,
                "amount_of_satellite_per_orbit_plane": 10,
                "inclination": 53.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": 15.05,
            },
            "orbital_connectivity_property": {
                "adjacent_inter_satellite_shifting": 0,
                "maximum_connected_ground_object": 10000,
                "maximum_connected_user_terminal": 1000,
                "maximum_connected_ground_station": 10,
            },
        }
    ],
}

simulation_properties = SimulationProperty(**lab_config)

## 2. Listen to the ground station link events
We keep one record per link, from its `GSL_CREATED` to its `GSL_REMOVED`:
- `on_gsl_created`: the link starts. We keep its satellite and its predicted disconnection time. If a link of the same Ground Station was removed in the same tick, this is a **handover**
- `on_gsl_updated`: we keep the last elevation of the link, to see how low the satellite was when it dropped
- `on_gsl_removed`: the link ends. We keep the time, and remember the removal until the end of the tick: if no new link follows, the Ground Station is **disconnected**

Satellites are named by their `topology_uniq_id`, as in the V4 lab, so both labs print the same handovers.

In [ ]:
links = {}
pending_removals = {}
handover_events = []


async def on_gsl_created(event: GSLCreatedEvent):
    links[event.link_id] = {
        "ground_station": event.ground_station_id,
        "satellite": event.satellite_id,
        "connected": event.sim_time,
        "predicted_disconnection": event.estimated_disconnection_time,
        "last_elevation": event.elevation_angle,
        "disconnected": None,
    }

    removal = pending_removals.pop(event.ground_station_id, None)
    if removal is None:
        print(f"[{event.sim_time:%H:%M:%S}] connected to Satellite#{event.satellite_id}")
        return
    handover_events.append(
        {"time": event.sim_time, "previous_satellite": removal["satellite"], "new_satellite": event.satellite_id}
    )
    print(f"[{event.sim_time:%H:%M:%S}] handover: Satellite#{removal['satellite']} -> Satellite#{event.satellite_id}")


async def on_gsl_updated(event: GSLUpdatedEvent):
    links[event.link_id]["last_elevation"] = event.elevation_above_horizon_angle


async def on_gsl_removed(event: GSLRemovedEvent):
    link = links[event.link_id]
    link["disconnected"] = event.sim_time
    pending_removals[event.ground_station_id] = link


def report_disconnections():
    """End of tick: a removal without a new link means no satellite in view."""
    for ground_station_id, removal in list(pending_removals.items()):
        handover_events.append(
            {"time": removal["disconnected"], "previous_satellite": removal["satellite"], "new_satellite": None}
        )
        print(f"[{removal['disconnected']:%H:%M:%S}] disconnected from Satellite#{removal['satellite']}, no satellite in view")
        del pending_removals[ground_station_id]


event_bus = EventBus()
event_bus.subscribe(EventType.GSL_CREATED, on_gsl_created)
event_bus.subscribe(EventType.GSL_UPDATED, on_gsl_updated)
event_bus.subscribe(EventType.GSL_REMOVED, on_gsl_removed)

## 3. Load and run the simulation
The bus is given to the configuration manager **before** `load_simulation()`: the first link is created while the simulation loads.

`run_synchronously` runs the simulation outside Jupyter's event loop, so each handler runs as soon as its event is published (see the note in [optical_terminal_lab.ipynb](optical_terminal_lab.ipynb)).

`report_disconnections` is the only action registered on the time manager: it runs at the end of each tick, after every event of the tick. No polling of the links.

In [ ]:
def run_synchronously(function, *args, **kwargs):
    """Run `function` outside Jupyter's event loop, so the event bus calls each handler right away."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(function, *args, **kwargs).result()


configuration_manager = BaseConfigurationManager(simulation_property=simulation_properties, event_bus=event_bus)
simulation_manager = run_synchronously(configuration_manager.load_simulation)
simulation_manager.time_manager.register_action(report_disconnections)

run_synchronously(simulation_manager.time_manager.tick_until_the_end, ticking_time_in_seconds=TICK_SECONDS)

## 4. Recap
### The handovers
The same list as the V4 lab: polling the links or listening to the events gives the same handovers.

In [ ]:
def describe_handover(event):
    new_satellite = (
        f"Satellite#{event['new_satellite']}" if event["new_satellite"] is not None else "no satellite in view"
    )
    return f"- {event['time']:%H:%M:%S} : Satellite#{event['previous_satellite']} -> {new_satellite}"


print(f"{len(handover_events)} handover(s) recorded")
for event in handover_events:
    print(describe_handover(event))

### Was the disconnection predicted?
When a link is created, the simulation predicts when the satellite will go below the 20° elevation of the Ground Station: `estimated_disconnection_time`. We compare it with the tick that actually removed the link.

The simulation only checks the links at each tick, so a link is removed at the **first tick after** the satellite went below 20°. If the prediction is right, the removal comes between 0 and 15 seconds (one tick) after the predicted time, and the last elevation seen by `GSL_UPDATED` is just above 20°.

In [ ]:
print(f"{'satellite':<14}{'connected':>10}{'predicted end':>15}{'removed':>10}{'removed - predicted':>21}{'last elevation':>16}")
for link in links.values():
    if link["disconnected"] is None:
        continue
    error_seconds = (link["disconnected"] - link["predicted_disconnection"]).total_seconds()
    within_a_tick = 0 <= error_seconds < TICK_SECONDS
    print(
        f"Satellite#{link['satellite']:<4}"
        + f"{link['connected']:%H:%M:%S}".rjust(10)
        + f"{link['predicted_disconnection']:%H:%M:%S}".rjust(15)
        + f"{link['disconnected']:%H:%M:%S}".rjust(10)
        + f"{error_seconds:+18.1f} s"
        + f"{link['last_elevation']:15.1f}°"
        + ("" if within_a_tick else "  <- outside one tick")
    )

still_up = [link for link in links.values() if link["disconnected"] is None]
for link in still_up:
    print(f"Satellite#{link['satellite']} still connected at the end, predicted end {link['predicted_disconnection']:%H:%M:%S}")

## What's next?
- Lower the tick to 5 seconds: the removal gets closer to the predicted time.
- An emulator does not need to wait for the removal: with `estimated_disconnection_time` it can prepare the next link in advance (make-before-break).
- `GSL_CREATED` also carries `estimated_duration`, the predicted lifetime of the link in seconds, and `GSL_UPDATED` refreshes the remaining one in `estimated_link_duration`.
- Combine with [inter_shell_cesium_lab.ipynb](inter_shell_cesium_lab.ipynb) to also follow the inter-satellite links with `ISL_*` events.